# EECE 490/690 – Applied Ensemble Learning: Random Forests and Boosting
**Dataset B – Heart Disease Prediction** · Member 2 section: Part 3 (Random Forest vs. boosting), Part 4 (class imbalance with SMOTENC)

Builds on Member 1's `shared_setup.py`: same split, same `CV` folds, same scoring, improved preprocessing.

## 0. Environment and shared setup

In [1]:
# In Colab, imbalanced-learn may need installing
%pip install -q imbalanced-learn xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import sklearn, imblearn, xgboost
warnings.filterwarnings("ignore", category=UserWarning)

print("python", sys.version.split()[0])
for m in (sklearn, imblearn, xgboost, pd, np):
    print(m.__name__, m.__version__)

python 3.12.10
sklearn 1.9.1
imblearn 0.14.2
xgboost 3.4.1
pandas 3.1.0
numpy 2.5.4


In [3]:
from shared_setup import *   # SEED, CV, SCORING, NUM_COLS, CAT_COLS, load_and_split, build_preprocessor, build_numeric_pipeline, summarize_cv
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None); pd.set_option("display.width", 200)

X_train, X_test, y_train, y_test = load_and_split("heart.csv")   # X_test / y_test are NOT used in this section
print("train:", X_train.shape, " test (locked):", X_test.shape)

train: (734, 11)  test (locked): (184, 11)


## Part 3 – Random Forests and Boosting
**Requirements:** RF, AdaBoost (defaults OK), XGBoost + Dummy reference; identical partitions (`CV`); tune 2 hyperparameters for RF and 2 for XGBoost, max 6 settings each; train + validation scores for every model.

### 3.0 Protocol
TODO (markdown): describe the protocol (data, folds, preprocessing, tuning budget) and justify the metric used to pick the best setting.

### 3.1 Random Forest – grid search (6 settings)

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_validate

# Same pipeline as Member 1's Part 2 RF (step names "pre"/"model", 300 trees) -> only max_depth / min_samples_leaf change
rf_pipe = Pipeline([
    ("pre", build_preprocessor("improved")),
    ("model", RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)),
])
rf_grid = {"model__max_depth": [None, 5, 10],      # 3 x 2 = 6 candidate settings
           "model__min_samples_leaf": [1, 5]}

rf_search = GridSearchCV(rf_pipe, rf_grid, cv=CV, scoring=SCORING,
                         refit="roc_auc",            # selection metric (recall alone is maxed by the dummy)
                         return_train_score=True)
rf_search.fit(X_train, y_train)                      # training data only; preprocessing refit inside every fold


def grid_table(search):
    """(a) One row per candidate, train_*/val_* as mean +/- std (same format as summarize_cv)."""
    res, rows = search.cv_results_, []
    for i, params in enumerate(res["params"]):
        row = {k.replace("model__", ""): str(v) for k, v in params.items()}
        for m in SCORING:
            for split, label in (("train", "train"), ("test", "val")):
                row[f"{label}_{m}"] = f"{res[f'mean_{split}_{m}'][i]:.3f} +/- {res[f'std_{split}_{m}'][i]:.3f}"
        row["rank_val_roc_auc"] = res["rank_test_roc_auc"][i]
        rows.append(row)
    return pd.DataFrame(rows).sort_values("rank_val_roc_auc").reset_index(drop=True)


rf_grid_table = grid_table(rf_search)
display(rf_grid_table)

# (b) Best setting re-scored with cross_validate on the same CV folds -> row in Member 1's exact format
print("Best RF params:", rf_search.best_params_, "| mean val ROC-AUC:", round(rf_search.best_score_, 3))
rf_best_cv = cross_validate(rf_search.best_estimator_, X_train, y_train, cv=CV,
                            scoring=SCORING, return_train_score=True)
rf_best_row = pd.DataFrame([summarize_cv(rf_best_cv, "RandomForest (tuned)")]).set_index("model")
rf_best_row

,max_depth,min_samples_leaf,train_accuracy,val_accuracy,train_precision,val_precision,train_recall,val_recall,train_f1,val_f1,train_roc_auc,val_roc_auc,rank_val_roc_auc
0,10,5,0.907 +/- 0.007,0.860 +/- 0.021,0.893 +/- 0.005,0.848 +/- 0.009,0.945 +/- 0.011,0.909 +/- 0.033,0.918 +/- 0.006,0.877 +/- 0.020,0.973 +/- 0.002,0.932 +/- 0.013,1
1,None,5,0.908 +/- 0.008,0.861 +/- 0.020,0.896 +/- 0.004,0.850 +/- 0.007,0.943 +/- 0.011,0.909 +/- 0.033,0.919 +/- 0.007,0.878 +/- 0.019,0.974 +/- 0.002,0.932 +/- 0.013,2
2,5,1,0.911 +/- 0.006,0.865 +/- 0.020,0.902 +/- 0.006,0.856 +/- 0.006,0.942 +/- 0.010,0.909 +/- 0.040,0.922 +/- 0.005,0.881 +/- 0.020,0.973 +/- 0.002,0.932 +/- 0.014,3
3,5,5,0.896 +/- 0.006,0.861 +/- 0.020,0.885 +/- 0.001,0.850 +/- 0.007,0.932 +/- 0.011,0.909 +/- 0.033,0.908 +/- 0.005,0.878 +/- 0.019,0.962 +/- 0.003,0.932 +/- 0.015,4
4,10,1,0.995 +/- 0.003,0.850 +/- 0.010,0.991 +/- 0.006,0.847 +/- 0.016,1.000 +/- 0.000,0.892 +/- 0.043,0.996 +/- 0.003,0.868 +/- 0.013,1.000 +/- 0.000,0.930 +/- 0.014,5
5,None,1,1.000 +/- 0.000,0.862 +/- 0.013,1.000 +/- 0.000,0.856 +/- 0.011,1.000 +/- 0.000,0.904 +/- 0.040,1.000 +/- 0.000,0.879 +/- 0.015,1.000 +/- 0.000,0.930 +/- 0.012,6


Best RF params: {'model__max_depth': 10, 'model__min_samples_leaf': 5} | mean val ROC-AUC: 0.932


,train_accuracy,val_accuracy,train_precision,val_precision,train_recall,val_recall,train_f1,val_f1,train_roc_auc,val_roc_auc
model,,,,,,,,,,
RandomForest (tuned),0.907 +/- 0.007,0.860 +/- 0.021,0.893 +/- 0.005,0.848 +/- 0.009,0.945 +/- 0.011,0.909 +/- 0.033,0.918 +/- 0.006,0.877 +/- 0.020,0.973 +/- 0.002,0.932 +/- 0.013


**Reading the Random Forest grid** (all numbers: mean over the 3 shared CV folds, training data only)

- **Sanity check.** The untuned setting `max_depth=None, min_samples_leaf=1` reproduces Member 1's improved-pipeline RF exactly (val accuracy 0.862, precision 0.856, recall 0.904, F1 0.879, ROC-AUC 0.930), so our pipeline and folds match Part 2.
- **Validation is flat.** Validation ROC-AUC lies between 0.930 and 0.932 across all six settings, so on this dataset the forest is insensitive to these two hyperparameters.
- **The difference is in the train–validation gap.** With `min_samples_leaf=1` and `max_depth` None or 10, train ROC-AUC is 1.000 (gap ≈ 0.07): the trees memorise the training folds. *Either* limit reduces this: shallower trees (`max_depth=5, min_samples_leaf=1`: train 0.973, gap ≈ 0.04) or bigger leaves (`min_samples_leaf=5`: gap ≈ 0.04 at depth None/10, 0.030 at depth 5). Validation ROC-AUC stays essentially the same (0.932 vs 0.930, well inside the fold std).
- **Why the overfit forests still validate well.** Each fully grown tree has low bias but high variance. Bootstrap samples and random feature subsets make the trees' errors partly independent, so averaging 300 of them cancels much of that variance. Bagging tolerates overfit base learners.
- **Recall (primary metric).** Validation recall is 0.909 for the top four settings, 0.904 for None / 1, and lowest for 10 / 1 (0.892). No setting trades recall for ROC-AUC in a meaningful way.
- **Selected setting.** Refit on ROC-AUC picks `max_depth=10, min_samples_leaf=5` (val ROC-AUC 0.932 ± 0.013, val recall 0.909 ± 0.033). However, the top four settings tie at 0.932 and the fold std is ≈ 0.013, so this is a statistical tie, not a clear winner; we keep it as a mildly regularised forest. The setting was also chosen on the same folds it is scored on (and the `cross_validate` row reuses those folds), so 0.932 is slightly optimistic; the unbiased estimate comes from the single test-set evaluation in Part 5.
- **Did extra complexity help?** No. Unrestricted trees reached train ROC-AUC 1.000 but validation 0.930, no better than the restricted settings.

### 3.2 XGBoost – grid search (6 settings)

In [5]:
from xgboost import XGBClassifier

# Same "pre" step and folds as the RF grid; n_estimators fixed at 200 -> only max_depth / learning_rate change
xgb_pipe = Pipeline([
    ("pre", build_preprocessor("improved")),
    ("model", XGBClassifier(n_estimators=200, eval_metric="logloss", random_state=SEED, n_jobs=-1)),
])
xgb_grid = {"model__max_depth": [3, 5],                 # 2 x 3 = 6 candidate settings
            "model__learning_rate": [0.05, 0.1, 0.3]}

xgb_search = GridSearchCV(xgb_pipe, xgb_grid, cv=CV, scoring=SCORING,
                          refit="roc_auc", return_train_score=True)
xgb_search.fit(X_train, y_train)                        # training data only

xgb_grid_table = grid_table(xgb_search)                 # (a) same helper as 3.1
display(xgb_grid_table)

# (b) best setting re-scored on the same folds, Member 1's row format
print("Best XGB params:", xgb_search.best_params_, "| mean val ROC-AUC:", round(xgb_search.best_score_, 3))
xgb_best_cv = cross_validate(xgb_search.best_estimator_, X_train, y_train, cv=CV,
                             scoring=SCORING, return_train_score=True)
xgb_best_row = pd.DataFrame([summarize_cv(xgb_best_cv, "XGBoost (tuned)")]).set_index("model")
xgb_best_row

,learning_rate,max_depth,train_accuracy,val_accuracy,train_precision,val_precision,train_recall,val_recall,train_f1,val_f1,train_roc_auc,val_roc_auc,rank_val_roc_auc
0,0.05,3,0.950 +/- 0.003,0.862 +/- 0.008,0.948 +/- 0.005,0.863 +/- 0.020,0.962 +/- 0.011,0.894 +/- 0.033,0.955 +/- 0.003,0.878 +/- 0.009,0.990 +/- 0.002,0.929 +/- 0.006,1
1,0.1,3,0.984 +/- 0.005,0.858 +/- 0.010,0.982 +/- 0.008,0.861 +/- 0.009,0.989 +/- 0.003,0.887 +/- 0.024,0.985 +/- 0.004,0.874 +/- 0.011,0.999 +/- 0.001,0.922 +/- 0.004,2
2,0.1,5,1.000 +/- 0.000,0.845 +/- 0.007,1.000 +/- 0.000,0.841 +/- 0.009,1.000 +/- 0.000,0.887 +/- 0.015,1.000 +/- 0.000,0.863 +/- 0.007,1.000 +/- 0.000,0.921 +/- 0.006,3
3,0.05,5,0.996 +/- 0.003,0.850 +/- 0.002,0.993 +/- 0.005,0.846 +/- 0.008,1.000 +/- 0.000,0.892 +/- 0.009,0.996 +/- 0.003,0.868 +/- 0.001,1.000 +/- 0.000,0.919 +/- 0.006,4
4,0.3,5,1.000 +/- 0.000,0.856 +/- 0.002,1.000 +/- 0.000,0.856 +/- 0.009,1.000 +/- 0.000,0.889 +/- 0.016,1.000 +/- 0.000,0.872 +/- 0.003,1.000 +/- 0.000,0.919 +/- 0.011,5
5,0.3,3,1.000 +/- 0.000,0.862 +/- 0.015,1.000 +/- 0.000,0.860 +/- 0.010,1.000 +/- 0.000,0.897 +/- 0.022,1.000 +/- 0.000,0.878 +/- 0.015,1.000 +/- 0.000,0.915 +/- 0.005,6


Best XGB params: {'model__learning_rate': 0.05, 'model__max_depth': 3} | mean val ROC-AUC: 0.929


,train_accuracy,val_accuracy,train_precision,val_precision,train_recall,val_recall,train_f1,val_f1,train_roc_auc,val_roc_auc
model,,,,,,,,,,
XGBoost (tuned),0.950 +/- 0.003,0.862 +/- 0.008,0.948 +/- 0.005,0.863 +/- 0.020,0.962 +/- 0.011,0.894 +/- 0.033,0.955 +/- 0.003,0.878 +/- 0.009,0.990 +/- 0.002,0.929 +/- 0.006


**Reading the XGBoost grid** (mean over the 3 shared CV folds, training data only)

- **Only the gentlest setting avoids memorising the training folds.** With `learning_rate=0.05, max_depth=3`, train ROC-AUC is 0.990; `0.1 / 3` already reaches 0.999 and the other four settings reach 1.000. The gentlest setting is also the one that validates best (val ROC-AUC 0.929 ± 0.006), while the most aggressive learning rate gives the lowest validation score (`0.3 / 3`: 0.915 ± 0.005). Unlike the forest, here overfitting the training folds *does* cost validation performance: each boosting round fits the previous rounds' residual errors, so with a large step size or deep trees the later rounds start fitting noise, and nothing averages it away.
- **Larger spread than Random Forest, but not conclusive.** Validation ROC-AUC ranges from 0.915 to 0.929 (a spread of 0.014, versus 0.930–0.932 for RF), and the fold std of each setting is 0.004–0.011. The ordering is consistent with the overfitting story, but with only 3 folds the differences between neighbouring settings are within noise.
- **Recall.** Validation recall stays between 0.887 and 0.897 for all six settings; the selected setting has 0.894 ± 0.033.
- **Selected setting and limitation.** Refit on ROC-AUC picks `learning_rate=0.05, max_depth=3`. This is at the *edge* of the grid (smallest learning rate, shallowest trees), so an even gentler setting (or more rounds at a smaller rate) might score higher. We do not expand the grid: the assignment limits tuning to six candidate settings, so we report this as a limitation.
- **Same-folds caveat.** As for RF, the setting was chosen on the same folds it is scored on (and the `cross_validate` row reuses those folds), so 0.929 is slightly optimistic; the unbiased estimate comes from the single test-set evaluation in Part 5.

In [6]:
# TODO: visualisation of the two grids (e.g. heatmap of validation score)

### 3.3 AdaBoost (default) and full comparison

In [7]:
# TODO: cross_validate Dummy, RF (default/tuned), AdaBoost, XGBoost (tuned) on CV with return_train_score=True; summarize_cv table

In [8]:
# TODO: train vs validation table / plot (overfitting check)

### 3.4 Interpretation and guiding questions
TODO (markdown):
- Comparison summary with numbers from the outputs above
- Why can boosting outperform bagging?
- When can boosting overfit?
- Did extra complexity help?

## Part 4 – Class Imbalance and SMOTENC
**Requirements:** report the class ratio; one controlled experiment (same model with vs. without SMOTENC) in an `imblearn` pipeline; categoricals handled correctly; resampling only inside training folds; discuss whether it was needed.

### 4.1 Class ratio

In [9]:
# TODO: class counts and ratio on y_train

TODO (markdown): what is the ratio and which class is the minority?

### 4.2 SMOTENC pipeline
TODO (markdown): explain the step order (impute + scale numerics → SMOTENC on un-encoded categoricals → one-hot → model) and why each step is there.

In [10]:
# TODO: build the imblearn Pipeline (reuse build_numeric_pipeline("improved")); sanity-check the resampled data

### 4.3 With vs. without SMOTENC

In [11]:
# TODO: cross_validate both pipelines (only SMOTENC differs) and compare

In [12]:
# TODO: visualisation of the comparison

### 4.4 Interpretation
TODO (markdown): did SMOTENC help? why? can recall improve while precision/accuracy worsen? was resampling needed?

### Handoff to Member 3
TODO (markdown): recommended final-model candidate (validation only), objects/tables Member 3 should use, test set still untouched.

In [13]:
# TODO: save all CV result tables to member2_cv_results.csv